In [0]:
%sql
-- CREATING EXTERNAL LOCATION WITH ADLS CREDENTIAL
CREATE STORAGE CREDENTIAL adls_cred
WITH AZURE_MANAGED_IDENTITY;

CREATE EXTERNAL LOCATION bronze_loc
URL "adlsse://<storage-account-name>.dfs.core.windows.net/bronze"
WITH (STORAGE CREDENTIAL adls_cred);

In [0]:
%sql
-- GRANTING PERMISSIONS TO THE USE USING GROUP APPROACH

GRANT SELECT, MODIFY
ON TBALE db_catalog.bronze.table_name
TO data_engineers;   -- HERE data_engineers IS A GROUP
    
-- GRANTING PERMISSIONS TO THE USE USING USER APPROACH
GRANT SELECT, MODIFY
ON TBALE db_catalog.silver.table_name
TO analyst;   -- HERE analyst IS A GROUP

In [0]:
%sql
-- Create function for masking column values(PII values)
create function mask_email(email string)
return 
case when is_account_group_member('data_engineers') then email
else "***"
end;

-- aplying masking to the table 
alter table db_catalog.silver.table_name 
alter column customer_email set mask mask_email;